# 🧠 PulseMind AI — Inspection Notebook
> *IronPulse Fitness Intelligence Engine — v2 with Adversarial Training & SHAP Explainability*

This notebook lets you explore the AI engine end-to-end:
1. **Data inspection** — your exercise library and workout history
2. **Feature extraction** — the 18 features the model uses
3. **Architecture details** — MLP, ResNet, AttentionNet, Ensemble
4. **Training with adversarial robustness** — FGSM + Mixup
5. **Explainability** — SHAP values and Integrated Gradients
6. **Weight visualisation** — first-layer heatmap


In [ ]:
import os, sys, importlib
import django
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Django setup ───────────────────────────────────────────
sys.path.insert(0, os.getcwd())
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'gymapp.settings')
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'
django.setup()

import torch

# Force reload so notebook picks up latest code changes
import core.ai.engine, core.ai.trainer
importlib.reload(core.ai.engine)
importlib.reload(core.ai.trainer)

print(f'PyTorch:  {torch.__version__}')
print(f'Platform: {sys.platform} / {os.uname().machine}')
print(f'Device:   CPU  (optimal for tabular models)')


## 1 — Data Inspection
Exercise library statistics from your IronPulse database.

In [ ]:
from core.models import Exercise, WorkoutSet, WorkoutSession

exercises = Exercise.objects.all().values('name', 'muscle_group', 'difficulty', 'is_compound', 'description')
df_ex = pd.DataFrame(list(exercises))

if df_ex.empty:
    print('No exercises found. Run: python manage.py import_wger')
else:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle('Exercise Library', fontsize=14, fontweight='bold')

    sns.countplot(data=df_ex, y='muscle_group', hue='muscle_group',
                  palette='viridis', ax=axes[0], legend=False,
                  order=df_ex['muscle_group'].value_counts().index)
    axes[0].set_title('By Muscle Group')

    sns.countplot(data=df_ex, x='difficulty', hue='difficulty',
                  palette='magma', ax=axes[1], legend=False)
    axes[1].set_title('By Difficulty')

    df_ex['type'] = df_ex['is_compound'].map({True: 'Compound', False: 'Isolation'})
    sns.countplot(data=df_ex, x='type', hue='type',
                  palette='coolwarm', ax=axes[2], legend=False)
    axes[2].set_title('Compound vs Isolation')

    plt.tight_layout()
    plt.show()

    print(f'Total exercises:   {len(df_ex)}')
    print(f'With descriptions: {df_ex["description"].apply(lambda x: bool(x)).sum()}')
    print(f'Sessions logged:   {WorkoutSession.objects.count()}')
    print(f'Total sets logged: {WorkoutSet.objects.count()}')


## 2 — Feature Extraction
The 18-dimensional feature vector extracted from each workout session.

In [ ]:
from core.ai.trainer import prepare_workout_data
from core.ai.engine import N_FEATURES, FEATURE_NAMES

sessions     = WorkoutSession.objects.all()
exercises_qs = Exercise.objects.all()

X, y = prepare_workout_data(sessions, exercises_qs)
print(f'Feature matrix X:  {X.shape}  (samples × {N_FEATURES} features)')
print(f'Label matrix y:    {y.shape}  (samples × exercises)')

n_real = sessions.count()
print(f'Real sessions:     {n_real}')
print(f'Synth augmented:   {max(0, len(X) - n_real)} (fallback when < 20 real sessions)')

# Feature statistics
df_feat = pd.DataFrame(X, columns=FEATURE_NAMES)

# Correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
corr = df_feat.corr()
sns.heatmap(corr, ax=axes[0], cmap='RdBu_r', center=0,
            xticklabels=True, yticklabels=True, square=True, linewidths=.3)
axes[0].set_title('Feature Correlation Matrix')
axes[0].tick_params(axis='x', rotation=45, labelsize=8)
axes[0].tick_params(axis='y', rotation=0, labelsize=8)

# Feature distributions
df_feat.boxplot(ax=axes[1], vert=True, rot=45)
axes[1].set_title('Feature Distributions (normalised)')
plt.tight_layout()
plt.show()
df_feat.describe().round(3)


## 3 — Model Architectures
Forward pass test + parameter counts for all four architectures.

In [ ]:
from core.ai.engine import build_model, MODEL_REGISTRY

n_classes = y.shape[1]

for arch_name in MODEL_REGISTRY:
    model = build_model(arch_name, input_size=N_FEATURES, output_size=n_classes)
    info  = model.get_architecture_info()

    model.eval()
    with torch.no_grad():
        out = model(torch.randn(1, N_FEATURES))
    probs = out.exp()

    print(f'\n{'='*55}')
    print(f'  {arch_name.upper():10s} | params: {info["total_params"]:,}')
    print(f'  Output shape: {tuple(out.shape)}  |  prob sum: {probs.sum().item():.5f}')
    for layer in info.get('layers', [])[:5]:
        name = layer.get('name', '')
        typ  = layer.get('type', '')
        inp  = layer.get('in', '')
        out_ = layer.get('out', '')
        print(f'    {name:<20} {typ:<15} {inp!s:>5} → {out_!s}')


## 4 — Training with Adversarial Robustness
FGSM adversarial training + Mixup augmentation on ResNet.

In [ ]:
from core.ai.trainer import PulseMindTrainer

model = build_model('resnet', input_size=N_FEATURES, output_size=n_classes)
print(f'Model: {model.architecture} | Params: {sum(p.numel() for p in model.parameters()):,}')

# 80/20 split
n = len(X)
idx    = np.random.permutation(n)
split  = int(0.8 * n)
X_tr, y_tr = X[idx[:split]], y[idx[:split]]
X_val, y_val = X[idx[split:]], y[idx[split:]]

# If not enough samples for a split, use all for both
if len(X_val) == 0:
    X_val, y_val = X_tr, y_tr

print(f'Train: {len(X_tr)} | Val: {len(X_val)}')

trainer = PulseMindTrainer(
    model, lr=3e-4, patience=20,
    use_adversarial=True,   # FGSM adversarial training
    use_mixup=True,         # Mixup augmentation
)

history = trainer.fit(X_tr, y_tr, X_val, y_val, epochs=80, verbose=True)

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs_axis = range(len(history['train_loss']))

axes[0].plot(epochs_axis, history['train_loss'], label='Train', color='#f97316', lw=1.5)
axes[0].plot(epochs_axis, history['val_loss'],   label='Val',   color='#6366f1', lw=1.5)
axes[0].plot(epochs_axis, history['adv_loss'],   label='Adv',   color='#ef4444', lw=1, linestyle='--', alpha=.7)
axes[0].set_title('Loss Curves (Train / Val / Adversarial)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('KL Divergence')
axes[0].legend()

axes[1].plot(epochs_axis, [a * 100 for a in history['accuracy']], color='#4ade80', lw=2)
axes[1].set_title('Top-3 Accuracy (%)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy %')
axes[1].set_ylim(0, 105)

plt.tight_layout()
plt.show()

print(f'Best epoch:   {history["best_epoch"]}')
print(f'Best val loss:{history["best_val_loss"]:.4f}')
print(f'Peak accuracy:{max(history["accuracy"])*100:.1f}%')


## 5 — Explainability
SHAP DeepExplainer (if installed) or Integrated Gradients — which features drive the AI.

In [ ]:
from core.ai.trainer import compute_shap_values, compute_integrated_gradients

# SHAP / IG attribution
shap_attr = compute_shap_values(model, X_tr, X_val)
ig_attr   = compute_integrated_gradients(
    model, torch.tensor(X_val[:10], dtype=torch.float32), steps=50
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# SHAP / gradient saliency
paired = sorted(zip(FEATURE_NAMES, shap_attr), key=lambda x: -x[1])
names, vals = zip(*paired)
colors = [f'hsla({200 + i*12},70%,60%,0.85)' for i in range(len(names))]

ax = axes[0]
bars = ax.barh(names, vals, color=plt.cm.plasma(np.linspace(0.2, 0.9, len(names))))
ax.set_title('SHAP / Gradient Saliency Attribution', fontweight='bold')
ax.set_xlabel('Mean |Attribution|')
ax.invert_yaxis()

# Integrated gradients
paired_ig = sorted(zip(FEATURE_NAMES, ig_attr), key=lambda x: -x[1])
ig_names, ig_vals = zip(*paired_ig)
axes[1].barh(ig_names, ig_vals, color=plt.cm.viridis(np.linspace(0.2, 0.9, len(ig_names))))
axes[1].set_title('Integrated Gradients Attribution', fontweight='bold')
axes[1].set_xlabel('Mean |IG|')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

print('Top 5 features by SHAP attribution:')
for name, val in list(paired)[:5]:
    bar = '█' * int(val * 50)
    print(f'  {name:22s}  {val:.4f}  {bar}')


## 6 — Weight Visualisation
First-layer weight matrix — what input patterns the model looks for.

In [ ]:
# Get first linear layer weights
first_linear = None
for module in model.modules():
    if isinstance(module, torch.nn.Linear):
        first_linear = module
        break

if first_linear is not None:
    W = first_linear.weight.detach().numpy()   # shape: (hidden, N_FEATURES)
    n_show = min(32, W.shape[0])

    plt.figure(figsize=(14, 6))
    sns.heatmap(
        W[:n_show, :],
        xticklabels=FEATURE_NAMES,
        yticklabels=[f'Neuron {i}' for i in range(n_show)],
        cmap='RdBu_r', center=0, linewidths=0.3, square=False
    )
    plt.title(f'{model.architecture} — First Linear Layer Weights ({n_show} × {N_FEATURES})')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
